# 1. old recommendation model code

In [1]:
# ========================== IMPORTS ==========================
import feedparser
import pandas as pd
import time
import re
import os
import random
import torch
import torch.nn.functional as F
import numpy as np
from transformers import AutoTokenizer, AutoModel
from datetime import datetime
from urllib.parse import urlencode
from sklearn.metrics.pairwise import cosine_similarity


# ========================== GLOBAL CONFIG ==========================
MAX_RESULTS_PER_CATEGORY = 100
MAX_TOTAL_PER_CATEGORY = 300
WAIT_TIME = 3
SAVE_PATH = "/content/arxiv_papers_with_metadata.csv"

# Categories for collection
ARXIV_CATEGORIES = {
    "machine learning": "cs.LG",
    "deep learning": "cs.CV",
    "natural language processing": "cs.CL",
    "reinforcement learning": "cs.AI",
    "cybersecurity": "cs.CR",
    "data science": "stat.ML",
    "quantum computing": "quant-ph",
    "internet of things": "cs.NI",
    "bioinformatics": "q-bio.BM",
    "graph theory": "math.CO",
    "probability": "math.PR",
    "statistics": "stat.TH",
    "augmented reality": "cs.HC",
    "virtual reality": "cs.HC",
    "robotics": "cs.RO",
    "algorithms": "cs.DS"
}


# ========================== UTILITY FUNCTIONS ==========================
def format_month_year(date_str):
    """Convert date string to 'Mon YYYY'."""
    try:
        date_obj = datetime.strptime(date_str, "%Y-%m-%dT%H:%M:%SZ")
        return date_obj.strftime("%b %Y")
    except Exception:
        return None


def extract_model_id(url):
    """Extract arXiv ID (without dots) from URL."""
    try:
        paper_id = url.split("/")[-1]
        return paper_id.replace(".", "")
    except Exception:
        return None


def safe_get(entry, key, default=None):
    """Safely extract metadata fields."""
    return getattr(entry, key, default) if hasattr(entry, key) else default


# ========================== STEP 1: COLLECT FROM ARXIV ==========================
def collect_from_arxiv():
    papers = []
    base_url = "http://export.arxiv.org/api/query?"

    for subcategory, cat_code in ARXIV_CATEGORIES.items():
        print(f"\n🔹 Collecting from arXiv for: {subcategory}")

        for start in range(0, MAX_TOTAL_PER_CATEGORY, MAX_RESULTS_PER_CATEGORY):
            query_params = {
                "search_query": f"cat:{cat_code}",
                "start": start,
                "max_results": MAX_RESULTS_PER_CATEGORY
            }

            url = base_url + urlencode(query_params)
            feed = feedparser.parse(url)

            for entry in feed.entries:
                title = safe_get(entry, "title", "").strip().replace("\n", " ")
                summary = safe_get(entry, "summary", "").strip().replace("\n", " ")
                authors = ", ".join(author.name for author in entry.authors) if "authors" in entry else "Unknown"
                published = safe_get(entry, "published", None)
                month_year = format_month_year(published) if published else None
                url_main = safe_get(entry, "id", None)
                pdf_url = entry.links[1].href if len(entry.links) > 1 else None

                doi = safe_get(entry, "arxiv_doi", None)
                journal_ref = safe_get(entry, "arxiv_journal_ref", None)
                comment = safe_get(entry, "arxiv_comment", None)
                primary_cat = entry.tags[0]["term"] if "tags" in entry and len(entry.tags) > 0 else None
                version = re.search(r'v(\d+)$', url_main).group(1) if url_main and re.search(r'v(\d+)$', url_main) else "1"

                model_id = extract_model_id(url_main)
                year = None
                if published:
                    try:
                        year = datetime.strptime(published, "%Y-%m-%dT%H:%M:%SZ").year
                    except Exception:
                        year = None

                papers.append({
                    "model_id": model_id,
                    "title": title,
                    "authors": authors,
                    "month_year": month_year,
                    "year": year,
                    "category": "Computer Science",
                    "subcategory": subcategory,
                    "source": "arxiv",
                    "abstract": summary,
                    "url": url_main,
                    "pdf_url": pdf_url,
                    "doi": doi,
                    "journal_ref": journal_ref,
                    "comment": comment,
                    "primary_category": primary_cat,
                    "version": version
                })

            print(f"   ✅ Fetched {len(feed.entries)} entries (start={start})")
            time.sleep(WAIT_TIME + random.uniform(0, 1))

    print("\n✅ Data collection complete.")
    return pd.DataFrame(papers)


# ========================== STEP 2: POPULARITY SCORING ==========================
def compute_popularity(df):
    """Compute metadata-based popularity score."""
    current_year = datetime.now().year
    df["year"] = df["year"].fillna(current_year - 5)

    # Recency normalization
    min_year, max_year = df["year"].min(), df["year"].max()
    df["recency_score"] = (df["year"] - min_year) / (max_year - min_year)

    # Author count
    df["author_count"] = df["authors"].apply(lambda x: len(x.split(",")) if isinstance(x, str) else 1)
    df["author_score"] = df["author_count"].apply(lambda x: min(x / 10, 1))

    # DOI and Journal presence
    df["doi_score"] = df["doi"].notnull().astype(int)
    df["journal_score"] = df["journal_ref"].notnull().astype(int)

    # Version activity
    df["version_count"] = df["version"].astype(int)
    df["version_score"] = df["version_count"].apply(lambda x: min(x / 5, 1))

    # Prestige keywords
    def prestige_score(comment):
        if not comment:
            return 0
        keywords = ["nature", "neurips", "icml", "cvpr", "science", "acl", "kdd"]
        return 1 if any(k in comment.lower() for k in keywords) else 0

    df["prestige_score"] = df["comment"].apply(prestige_score)

    # Weighted popularity
    df["popularity_score"] = (
        0.4 * df["recency_score"]
        + 0.2 * df["author_score"]
        + 0.15 * df["doi_score"]
        + 0.15 * df["journal_score"]
        + 0.05 * df["version_score"]
        + 0.05 * df["prestige_score"]
    )

    print("✅ Popularity scores computed.")
    return df


# ========================== STEP 3: EMBEDDING CREATION ==========================
def generate_embeddings(df, model_name="sentence-transformers/all-MiniLM-L6-v2"):
    """Generate sentence embeddings for all papers."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    print(f"✅ Model loaded on {device}")

    def mean_pooling(model_output, attention_mask):
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    batch_size = 32
    all_embeddings = []

    for i in range(0, len(df), batch_size):
        batch_texts = df["text_for_embedding"].iloc[i:i+batch_size].tolist()
        encoded_input = tokenizer(batch_texts, padding=True, truncation=True, max_length=256, return_tensors='pt').to(device)

        with torch.no_grad():
            model_output = model(**encoded_input)

        sentence_embeddings = mean_pooling(model_output, encoded_input["attention_mask"])
        sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)
        all_embeddings.append(sentence_embeddings.cpu().numpy())

    embeddings = np.vstack(all_embeddings)
    print("✅ Embeddings created:", embeddings.shape)
    return embeddings


# ========================== STEP 4: RECOMMENDATION FUNCTIONS ==========================
def recommend_by_category(df, embeddings, category_query, top_k=5):
    """Recommend top-K papers in a given category."""
    subset = df[df["subcategory"].str.contains(category_query, case=False, na=False)]
    if subset.empty:
        print("❌ No papers found in that category.")
        return None

    subset_indices = subset.index
    subset_embeddings = embeddings[subset_indices]
    centroid = np.mean(subset_embeddings, axis=0).reshape(1, -1)

    sim_scores = cosine_similarity(centroid, subset_embeddings).flatten()
    popularity_scores = subset["popularity_score"].fillna(0).values
    final_scores = 0.8 * sim_scores + 0.2 * popularity_scores

    top_indices = np.argsort(final_scores)[::-1][:top_k]
    top_papers = subset.iloc[top_indices][[
        "model_id", "title", "authors", "subcategory", "year", "popularity_score", "url"
    ]].copy()

    top_papers["semantic_similarity"] = sim_scores[top_indices]
    top_papers["final_score"] = final_scores[top_indices]
    return top_papers.sort_values(by="final_score", ascending=False)


def recommend_by_query_text(df, embeddings, tokenizer, model, query_text, top_k=5, alpha=0.8, beta=0.2):
    """Recommend top-k papers semantically similar to a query."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Encode query text and move all tensors to same device as model
    encoded_input = tokenizer([query_text], padding=True, truncation=True, return_tensors='pt')
    encoded_input = {k: v.to(model.device) for k, v in encoded_input.items()}

    with torch.no_grad():
        model_output = model(**encoded_input)

    token_embeddings = model_output[0]
    input_mask_expanded = encoded_input["attention_mask"].unsqueeze(-1).expand(token_embeddings.size()).float()
    query_embedding = torch.sum(token_embeddings * input_mask_expanded, 1) / \
                      torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    query_embedding = F.normalize(query_embedding, p=2, dim=1)

    # Move back to CPU for sklearn
    query_embedding_np = query_embedding.cpu().numpy()
    embeddings_np = embeddings  # already on CPU

    # Compute cosine similarity
    sim_scores = cosine_similarity(query_embedding_np, embeddings_np).flatten()

    # Combine semantic and popularity scores
    popularity_scores = df["popularity_score"].fillna(0).values
    final_scores = alpha * sim_scores + beta * popularity_scores

    # Pick top-k recommendations
    top_indices = np.argsort(final_scores)[::-1][:top_k]
    recommendations = df.iloc[top_indices][[
        "model_id", "title", "authors", "subcategory", "year", "popularity_score", "url"
    ]].copy()

    recommendations["semantic_similarity"] = sim_scores[top_indices]
    recommendations["final_score"] = final_scores[top_indices]
    return recommendations


# ========================== MAIN EXECUTION ==========================
if __name__ == "__main__":
    # Step 1: Collect and save papers
    # collect only if "/content/arxiv_papers_with_metadata.csv" not found
    if not os.path.exists(SAVE_PATH):
        df = collect_from_arxiv()
        df = compute_popularity(df)
        df.drop_duplicates(subset=["title"], inplace=True)
        df.to_csv(SAVE_PATH, index=False)
        print(f"\n📁 Saved to: {SAVE_PATH}")
        print(f"📊 Total Papers Collected: {len(df)}")
    else:
        df = pd.read_csv(SAVE_PATH)
        print(f"\n📁 Loaded from: {SAVE_PATH}")
        print(f"📊 Total Papers Collected: {len(df)}")

    # Step 2: Load and preprocess for embeddings
    df["title"] = df["title"].fillna("")
    df["abstract"] = df["abstract"].fillna("")
    df["subcategory"] = df["subcategory"].fillna("")
    df["text_for_embedding"] = df["title"] + ". " + df["abstract"]

    # Step 3: Generate embeddings
    embeddings = generate_embeddings(df)
    np.save("/content/arxiv_semantic_embeddings.npy", embeddings)

    # Step 4: Load tokenizer/model for query-based recommendation
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    model.eval()

    # Step 5: Example query
    query = "Attention is All you Need"
    print(f"\n🔍 Recommendations for: '{query}'\n")
    recs = recommend_by_query_text(df, embeddings, tokenizer, model, query, top_k=5)
    print(recs)


🔹 Collecting from arXiv for: machine learning
   ✅ Fetched 100 entries (start=0)
   ✅ Fetched 100 entries (start=100)


KeyboardInterrupt: 

# recompute_popularity_scores

In [2]:
import sqlite3
from datetime import datetime
import math

DB_PATH = "app/scholarlens.db"


def normalize(value, min_val, max_val):
    if value is None or min_val is None or max_val is None or max_val == min_val:
        return 0
    return (value - min_val) / (max_val - min_val)


def compute_popularity(row, min_year, max_year):
    # --------------------------
    # 1. Recency Score
    # --------------------------
    pub_year = None
    if row["published_at"]:
        try:
            pub_year = datetime.strptime(row["published_at"], "%Y-%m-%d").year
        except:
            pub_year = None

    recency_score = normalize(pub_year, min_year, max_year)

    # --------------------------
    # 2. Author Score
    # --------------------------
    authors = row["authors"] or ""
    author_count = authors.count(",") + 1 if authors else 1
    author_score = min(author_count / 10, 1.0)

    # --------------------------
    # 3. Abstract Score
    # --------------------------
    abstract = row["abstract"] or ""
    abstract_score = min(len(abstract) / 800.0, 1.0)

    # --------------------------
    # 4. Source Score
    # --------------------------
    source = (row["source"] or "").lower()

    if source == "ieee":
        source_score = 1.0
    elif source == "anthropic":
        source_score = 0.9
    elif source == "arxiv":
        source_score = 0.8
    else:
        source_score = 0.4

    # --------------------------
    # 5. Must Read Score
    # --------------------------
    is_must_read = row["is_must_read"] or 0
    must_read_score = 1.0 if is_must_read == 1 else 0.0

    # --------------------------
    # Final Weighted Score
    # --------------------------
    popularity_score = (
        0.45 * recency_score +
        0.20 * author_score +
        0.15 * source_score +
        0.10 * abstract_score +
        0.10 * must_read_score
    )

    return round(popularity_score, 5)


def recalc_popularity_scores():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row   # Return rows as dictionaries
    cur = conn.cursor()

    print("\n🔄 Loading papers from database...")

    cur.execute("SELECT * FROM unified_papers")
    rows = cur.fetchall()

    if not rows:
        print("⚠️ No papers found in unified_papers.")
        return

    # Extract valid years for normalization
    years = []
    for row in rows:
        if row["published_at"]:
            try:
                y = datetime.strptime(row["published_at"], "%Y-%m-%d").year
                years.append(y)
            except:
                pass

    min_year = min(years) if years else None
    max_year = max(years) if years else None

    print(f"📅 Year range: {min_year} → {max_year}")

    updated = 0

    print("\n⚙️ Recomputing popularity scores...\n")

    for row in rows:
        new_score = compute_popularity(row, min_year, max_year)

        cur.execute("""
            UPDATE unified_papers 
            SET popularity_score = ?
            WHERE paper_id = ?
        """, (new_score, row["paper_id"]))

        updated += 1

        if updated % 200 == 0:
            print(f"   ✔ Updated {updated} papers...")

    conn.commit()
    conn.close()

    print("\n✅ Popularity score recalculation complete!")
    print(f"📊 Total papers updated: {updated}\n")



In [3]:
recalc_popularity_scores()


🔄 Loading papers from database...
📅 Year range: 1965 → 2025

⚙️ Recomputing popularity scores...

   ✔ Updated 200 papers...
   ✔ Updated 400 papers...
   ✔ Updated 600 papers...
   ✔ Updated 800 papers...
   ✔ Updated 1000 papers...
   ✔ Updated 1200 papers...
   ✔ Updated 1400 papers...
   ✔ Updated 1600 papers...
   ✔ Updated 1800 papers...
   ✔ Updated 2000 papers...
   ✔ Updated 2200 papers...
   ✔ Updated 2400 papers...
   ✔ Updated 2600 papers...
   ✔ Updated 2800 papers...
   ✔ Updated 3000 papers...
   ✔ Updated 3200 papers...
   ✔ Updated 3400 papers...
   ✔ Updated 3600 papers...
   ✔ Updated 3800 papers...
   ✔ Updated 4000 papers...
   ✔ Updated 4200 papers...

✅ Popularity score recalculation complete!
📊 Total papers updated: 4221



# Specter2 embedding + recommendation

In [6]:
# !pip install -U adapters

In [ ]:
# ===========================================
# IMPORTS
# ===========================================
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime

import torch
import torch.nn.functional as F

from transformers import AutoTokenizer
from adapters import AutoAdapterModel
from sklearn.metrics.pairwise import cosine_similarity


DB_PATH = "/content/scholalens.db"   # Update if needed


# ===========================================
# LOAD PAPERS FROM DATABASE
# ===========================================
def load_papers():
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query("""
        SELECT paper_id, title, abstract, category, url, pdf_url, popularity_score
        FROM unified_papers
        WHERE title IS NOT NULL AND abstract IS NOT NULL
    """, conn)
    conn.close()
    return df


# ===========================================
# INIT SPECTER2 MODEL
# ===========================================
def load_specter2():
    print("Loading SPECTER2 model...")

    tokenizer = AutoTokenizer.from_pretrained("allenai/specter2_base")
    model = AutoAdapterModel.from_pretrained("allenai/specter2_base")

    # Load proximity adapter
    model.load_adapter("allenai/specter2", source="hf", load_as="specter2", set_active=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    print(f"Model loaded on {device}")
    return tokenizer, model, device


# ===========================================
# GENERATE EMBEDDINGS
# ===========================================
def generate_embeddings_specter2(df, tokenizer, model, device):
    print("Generating embeddings using SPECTER2...")

    batch_size = 16
    all_embeddings = []

    for i in range(0, len(df), batch_size):
        batch = df.iloc[i:i+batch_size]
        text_batch = [
            (row["title"] or "") + tokenizer.sep_token + (row["abstract"] or "")
            for _, row in batch.iterrows()
        ]

        inputs = tokenizer(
            text_batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
            return_token_type_ids=False
        ).to(device)

        with torch.no_grad():
            output = model(**inputs)

        # CLS token embedding
        embeddings = output.last_hidden_state[:, 0, :]
        embeddings = F.normalize(embeddings, p=2, dim=1)

        all_embeddings.append(embeddings.cpu().numpy())

    final_embeddings = np.vstack(all_embeddings)
    print("Embeddings shape:", final_embeddings.shape)
    return final_embeddings


# ===========================================
# SAVE EMBEDDINGS TO DATABASE
# ===========================================
def save_embeddings_to_db(df, embeddings, model_name="specter2"):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    print("Saving embeddings to database...")

    cursor.execute("DELETE FROM embeddings")  # Clear old embeddings

    for i, row in df.iterrows():
        emb_json = json.dumps(embeddings[i].tolist())
        cursor.execute("""
            INSERT INTO embeddings (paper_id, embedding, model_name, updated_at)
            VALUES (?, ?, ?, ?)
        """, (
            row["paper_id"],
            emb_json,
            model_name,
            datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        ))

    conn.commit()
    conn.close()
    print("✅ Embeddings saved into database!")


# ===========================================
# RECOMMENDATION: CATEGORY-BASED
# ===========================================
def recommend_by_category(category, df, embeddings, top_k=10):
    mask = df["category"].str.lower() == category.lower()
    subset = df[mask]

    if subset.empty:
        print("No papers in this category.")
        return None

    idx = subset.index
    subset_emb = embeddings[idx]

    centroid = np.mean(subset_emb, axis=0).reshape(1, -1)
    sim = cosine_similarity(centroid, subset_emb).flatten()

    pop = subset["popularity_score"].fillna(0).values

    final_score = 0.8 * sim + 0.2 * pop

    top_idx = np.argsort(final_score)[::-1][:top_k]
    recs = subset.iloc[top_idx].copy()
    recs["semantic_similarity"] = sim[top_idx]
    recs["final_score"] = final_score[top_idx]

    return recs.sort_values("final_score", ascending=False)


# ===========================================
# RECOMMENDATION: QUERY-TEXT
# ===========================================
def recommend_by_query(query, df, embeddings, tokenizer, model, device, top_k=10):
    encoded = tokenizer(
        [query],
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt",
        return_token_type_ids=False
    ).to(device)

    with torch.no_grad():
        out = model(**encoded)

    query_emb = out.last_hidden_state[:, 0, :]
    query_emb = F.normalize(query_emb, p=2, dim=1).cpu().numpy()

    sim_scores = cosine_similarity(query_emb, embeddings).flatten()
    pop = df["popularity_score"].fillna(0).values

    final = 0.8 * sim_scores + 0.2 * pop

    idx = np.argsort(final)[::-1][:top_k]

    recs = df.iloc[idx].copy()
    recs["semantic_similarity"] = sim_scores[idx]
    recs["final_score"] = final[idx]

    return recs.sort_values("final_score", ascending=False)


# ===========================================
# MAIN EXECUTION (FOR JUPYTER NOTEBOOK)
# ===========================================
if __name__ == "__main__":
    df = load_papers()
    tokenizer, model, device = load_specter2()

    embeddings = generate_embeddings_specter2(df, tokenizer, model, device)

    save_embeddings_to_db(df, embeddings)

    print("Model ready for recommendations!")


In [ ]:
df = load_papers()
tokenizer, model, device = load_specter2()
emb = generate_embeddings_specter2(df, tokenizer, model, device)

In [ ]:
recommend_by_category("deep learning", df, emb, top_k=10)


In [ ]:
recommend_by_query("graph neural networks for recommendation", df, emb, tokenizer, model, device, top_k=10)
